<a href="https://www.kaggle.com/code/kaiyoo88/rocketlaunch-simulation-benchmark-working-memory?scriptVersionId=311961162" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🚀 Getting Started with Kaggle Benchmarks

Welcome! This notebook will teach you how to create, run, and evaluate LLM benchmarks using the `kaggle-benchmarks` library.

**Key concepts** 
1. Task: A Python function defining the problem (e.g., "Solve this riddle").
2. Run: The execution of a task
3. Benchmark: A collection of tasks that is arbitrarily put together by a user. There is no code implementation for this. This is a feature that Kaggle supports on the graphical user interface so that users can put together their own benchmarks based on the tasks that they care about

Now, let's dive into creating a task and executing your first run!


In [ ]:
# %choose batch_riddle_solver

# Static RocketLaunch Simulation - Task: Working Memory

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import List

import pandas as pd
import kaggle_benchmarks as kbench


# =========================================================
# CONFIG
# =========================================================
VERBOSE = True

TASK_FILE = "/kaggle/input/datasets/kaiyoo88/task-json/TASK_JSON.json"

PLAN_HORIZON = 5
HISTORY_WINDOW = 3

# False: build task / leaderboard mode
# True : notebook analysis mode (compute memory_gain too)
ANALYSIS_MODE = False

MEMORY_HIDE_FIELDS = {
    "is_stage1_attached",
    "is_stage2_attached",
    "is_stage3_attached",
    "is_stage4_attached",
    "is_stage5_attached",
    "is_return_module_attached",
    "s4_separated",
    "s5_separated",
    "tube_flow_state",
    "s4_tube_locked",
    "s5_tube_locked",
    "sample_tube_locked",
    "is_in_earth_orbit",
    "is_outside_earth_escape",
    "is_outside_europa_escape",
    "launch_ready",
    "separation_ready",
    "mission_phase_match",
}

W_VALID = 0.15
W_IMMEDIATE = 0.20
W_PREFIX = 0.30
W_EDIT = 0.35


# =========================================================
# LOAD TASKS
# =========================================================
with open(TASK_FILE, "r", encoding="utf-8") as f:
    TASKS = json.load(f)

steps = TASKS


# =========================================================
# SCHEMA
# =========================================================
@dataclass
class PlanOutput:
    plan: List[str]


# =========================================================
# ACTION CATALOG
# =========================================================
FUNCTION_TABLE = [
    ("launch_from_earth", "Start launch from Earth."),
    ("wait", "Wait without taking immediate action and keep the current state."),
    ("abort_mission", "Abort the entire mission."),
    ("continue_burn", "Continue the current propulsion burn."),
    ("reduce_throttle", "Reduce thrust to manage fuel use or attitude."),
    ("shutdown_engine", "Shut down the current engine."),
    ("separate_stage1", "Separate Stage 1."),
    ("coast_transfer", "Continue transfer/movement by coasting without thrust."),
    ("trajectory_correction_burn", "Perform a short burn for trajectory correction."),
    ("restart_stage2_engine", "Restart the Stage 2 engine."),
    ("abort_capture", "Abort the attempt to capture into the target orbit."),
    ("separate_stage2", "Separate Stage 2."),
    ("continue_orbit_hold", "Maintain the current orbit."),
    ("adjust_stage3_center_of_mass", "Adjust the center of mass for Stage 3 landing."),
    ("adjust_stage3_booster_usage", "Adjust booster usage for Stage 3 landing."),
    ("continue_descent", "Continue the current descent."),
    ("abort_landing", "Abort the current landing attempt."),
    ("continue_ice_breaking", "Continue the ice-breaking operation."),
    ("pause_ice_breaking", "Pause the ice-breaking operation."),
    ("retrieve_ice_robot", "Retrieve the ice-work robot."),
    ("insert_pipe", "Insert the pipe."),
    ("realign_pipe", "Realign the pipe."),
    ("start_water_lift", "Start lifting water."),
    ("remove_pipe", "Remove the inserted pipe."),
    ("start_electrolysis", "Start electrolysis."),
    ("abort_surface_process", "Abort the surface resource processing operation."),
    ("continue_fill_process", "Continue the fuel/sample filling process."),
    ("route_sample_to_return_module", "Route the sample toward the return module."),
    ("lock_s4_tube", "Lock the Stage 4 line."),
    ("lock_s5_tube", "Lock the Stage 5 line."),
    ("lock_sample_tube", "Lock the sample line."),
    ("lock_all_tubes", "Lock all major lines at once."),
    ("launch_return_stack", "Launch the return stack from the surface of Europa."),
    ("abort_return_launch", "Abort the launch departing from Europa."),
    ("adjust_stage4_booster_usage", "Adjust Stage 4 booster usage during Europa escape."),
    ("adjust_stage4_center_of_mass", "Adjust the Stage 4 center of mass during Europa escape."),
    ("separate_stage4", "Separate Stage 4."),
    ("separate_stage5", "Separate Stage 5."),
    ("continue_coast", "Continue the current unpowered coast."),
    ("ignite_return_module_booster", "Ignite the return module booster."),
    ("shutdown_return_module_booster", "Shut down the return module booster."),
    ("adjust_return_module_center_of_mass", "Adjust the center of mass for return module landing."),
    ("adjust_return_module_booster_usage", "Adjust booster usage for return module landing."),
]

SEED = 2026
shuffled_functions = FUNCTION_TABLE[:]
rng = random.Random(SEED)
rng.shuffle(shuffled_functions)

ACTION_CATALOG = [
    {"id": f"A{i:02d}", "name": name, "desc": desc}
    for i, (name, desc) in enumerate(shuffled_functions, start=1)
]

NAME2ID = {a["name"]: a["id"] for a in ACTION_CATALOG}
ID2NAME = {a["id"]: a["name"] for a in ACTION_CATALOG}
ALL_IDS = [a["id"] for a in ACTION_CATALOG]


# =========================================================
# VALIDATION
# =========================================================
def validate_action_coverage(steps):
    catalog_names = set(NAME2ID.keys())

    missing_correct = sorted({
        step["correct_action"]
        for step in steps
        if step["correct_action"] not in catalog_names
    })

    missing_valid = sorted({
        action
        for step in steps
        for action in step["valid_actions"]
        if action not in catalog_names
    })

    print("=== ACTION COVERAGE CHECK ===")
    print("Missing correct_action:", missing_correct)
    print("Missing valid_actions :", missing_valid)

    if missing_correct or missing_valid:
        raise ValueError("ACTION_CATALOG is missing actions used in task_json.")


# =========================================================
# STATE HELPERS
# =========================================================
def sanitize_state(state: dict) -> dict:
    return dict(state)

def sanitize_state_for_memory_test(state: dict) -> dict:
    s = dict(state)
    for key in MEMORY_HIDE_FIELDS:
        s.pop(key, None)
    return s


# =========================================================
# GOLD PLAN
# =========================================================
def get_gold_plan(steps, start_idx: int) -> List[str]:
    gold_names = [steps[j]["correct_action"] for j in range(start_idx, len(steps))]
    if PLAN_HORIZON is not None:
        gold_names = gold_names[:PLAN_HORIZON]
    return [NAME2ID[name] for name in gold_names]


# =========================================================
# DISTANCE HELPERS
# =========================================================
def levenshtein(seq1: List[str], seq2: List[str]) -> int:
    n, m = len(seq1), len(seq2)
    if n == 0:
        return m
    if m == 0:
        return n

    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[n][m]

def prefix_match_len(pred: List[str], gold: List[str]) -> int:
    k = 0
    for p, g in zip(pred, gold):
        if p == g:
            k += 1
        else:
            break
    return k


# =========================================================
# SCORING
# =========================================================
def compute_valid_score(pred_ids: List[str], steps: List[dict], start_idx: int, horizon_len: int) -> float:
    if horizon_len == 0:
        return 0.0

    valid_hits = 0
    for k in range(horizon_len):
        if k >= len(pred_ids):
            break
        pred_id = pred_ids[k]
        if pred_id not in ID2NAME:
            continue
        pred_action_name = ID2NAME[pred_id]
        future_step = steps[start_idx + k]
        if pred_action_name in future_step["valid_actions"]:
            valid_hits += 1

    return valid_hits / horizon_len


def score_plan(pred_ids: List[str], gold_ids: List[str], steps: List[dict], start_idx: int):
    if len(gold_ids) == 0:
        return {
            "valid_score": 0.0,
            "immediate": 0.0,
            "prefix": 0.0,
            "edit": 0.0,
            "combined": 0.0,
        }

    pred_ids = [x for x in pred_ids if x in ALL_IDS]
    pred_ids = pred_ids[:len(gold_ids)]

    valid_score = compute_valid_score(
        pred_ids=pred_ids,
        steps=steps,
        start_idx=start_idx,
        horizon_len=len(gold_ids),
    )

    immediate = 1.0 if len(pred_ids) > 0 and pred_ids[0] == gold_ids[0] else 0.0
    prefix = prefix_match_len(pred_ids, gold_ids) / len(gold_ids)
    dist = levenshtein(pred_ids, gold_ids)
    edit = 1.0 - dist / max(len(pred_ids), len(gold_ids), 1)

    combined = (
        W_VALID * valid_score +
        W_IMMEDIATE * immediate +
        W_PREFIX * prefix +
        W_EDIT * edit
    )

    return {
        "valid_score": valid_score,
        "immediate": immediate,
        "prefix": prefix,
        "edit": edit,
        "combined": combined,
    }


# =========================================================
# HISTORY CONTEXT
# =========================================================
def build_history_context(steps, current_idx: int, window: int = 3):
    start = max(0, current_idx - window)
    history = []
    for j in range(start, current_idx):
        history.append({
            "step_index": j + 1,
            "task_id": steps[j]["task_id"],
            "state": sanitize_state(steps[j]["state"]),
            "action_id": NAME2ID[steps[j]["correct_action"]],
        })
    return {
        "recent_history": history,
        "action_sequence": [x["action_id"] for x in history],
    }


# =========================================================
# PROMPT
# =========================================================
def build_prompt(step: dict, remaining_steps: int, context=None, memory_mode=False) -> str:
    state = sanitize_state_for_memory_test(step["state"]) if memory_mode else sanitize_state(step["state"])

    if PLAN_HORIZON is None:
        horizon_text = (
            f"Return the full remaining action plan from now to mission completion. "
            f"There are {remaining_steps} remaining actions including the current one."
        )
    else:
        horizon_text = (
            f"Return the next {min(PLAN_HORIZON, remaining_steps)} actions starting from the current step."
        )

    context_block = ""
    if context is not None:
        context_block = (
            "\nHISTORY CONTEXT:\n"
            + json.dumps(context, ensure_ascii=False, indent=2)
            + "\n"
        )

    return f"""
You are a mission planner for a Europa resource-return mission.

Global mission objective:
Reach Europa, extract hydrogen, and return safely to Earth.

Task:
Predict the remaining action plan as a sequence of action IDs.

{horizon_text}

Rules:
- Use only action IDs from the action catalog.
- The first action must be the best immediate next action.
- Return JSON only.

Return JSON:
{{
  "plan": ["Axx", "Ayy", "Azz"]
}}
{context_block}
CURRENT STATE:
{json.dumps(state, ensure_ascii=False, indent=2)}

ACTION CATALOG:
{json.dumps([{"id": a["id"], "description": a["desc"]} for a in ACTION_CATALOG], ensure_ascii=False, indent=2)}
""".strip()


# =========================================================
# CORE
# =========================================================
def run_working_memory_episode(llm, steps, task_label="working_memory"):
    total = len(steps)

    weighted_sum_hist = 0.0
    weighted_sum_no_ctx = 0.0
    weight_total = 0.0
    logs = []

    with kbench.chats.new(f"{task_label}-chat"):
        for i, step in enumerate(steps):
            gold_plan = get_gold_plan(steps, i)
            step_weight = len(gold_plan)

            # =====================================================
            # HISTORY MODE (always needed)
            # =====================================================
            history_ctx = build_history_context(steps, i, HISTORY_WINDOW) 
            prompt_hist = build_prompt(
                step=step,
                remaining_steps=(len(steps) - i),
                context=history_ctx,
                memory_mode=True,
            )

            try:
                resp_hist = llm.prompt(prompt_hist, schema=PlanOutput)
                pred_hist = resp_hist.plan if resp_hist is not None and hasattr(resp_hist, "plan") and resp_hist.plan is not None else []
            except Exception as e:
                pred_hist = []
                if VERBOSE:
                    print(f"[HIST ][{i+1:02d}/{total:02d}] prompt/parsing error: {e}")

            scores_hist = score_plan(
                pred_ids=pred_hist,
                gold_ids=gold_plan,
                steps=steps,
                start_idx=i,
            )

            weighted_sum_hist += scores_hist["combined"] * step_weight

            # =====================================================
            # ANALYSIS MODE ONLY: NO-CONTEXT BASELINE
            # =====================================================
            if ANALYSIS_MODE:
                prompt_no = build_prompt(
                    step=step,
                    remaining_steps=(len(steps) - i),
                    context=None,
                    memory_mode=False,
                )
                try:
                    resp_no = llm.prompt(prompt_no, schema=PlanOutput)
                    pred_no = resp_no.plan if resp_no is not None and hasattr(resp_no, "plan") and resp_no.plan is not None else []
                except Exception as e:
                    pred_no = []
                    if VERBOSE:
                        print(f"[NOCTX][{i+1:02d}/{total:02d}] prompt/parsing error: {e}")

                scores_no = score_plan(
                    pred_ids=pred_no,
                    gold_ids=gold_plan,
                    steps=steps,
                    start_idx=i,
                )

                weighted_sum_no_ctx += scores_no["combined"] * step_weight
            else:
                pred_no = []
                scores_no = None

            weight_total += step_weight

            row = {
                "step": i + 1,
                "task_id": step["task_id"],
                "gold_len": len(gold_plan),
                "hist_score": scores_hist["combined"],
                "hist_first": pred_hist[0] if pred_hist else None,
                "gold_first": gold_plan[0] if gold_plan else None,
                "running_hist": weighted_sum_hist / weight_total if weight_total > 0 else 0.0,
            }

            if ANALYSIS_MODE:
                row.update({
                    "no_ctx_score": scores_no["combined"],
                    "no_ctx_first": pred_no[0] if pred_no else None,
                    "delta": scores_hist["combined"] - scores_no["combined"],
                    "running_no_ctx": weighted_sum_no_ctx / weight_total if weight_total > 0 else 0.0,
                })

            logs.append(row)

            if VERBOSE:
                if ANALYSIS_MODE:
                    print(
                        f"[{i+1:02d}/{total:02d}] "
                        f"NO={scores_no['combined']:.3f} "
                        f"HIST={scores_hist['combined']:.3f} "
                        f"Δ={row['delta']:+.3f} | "
                        f"pred0_no={row['no_ctx_first']} "
                        f"pred0_hist={row['hist_first']} "
                        f"gold0={row['gold_first']}"
                    )
                else:
                    print(
                        f"[{i+1:02d}/{total:02d}] "
                        f"HIST={scores_hist['combined']:.3f} | "
                        f"run={row['running_hist']:.3f} | "
                        f"pred0_hist={row['hist_first']} "
                        f"gold0={row['gold_first']}"
                    )

    planning_with_history = weighted_sum_hist / weight_total if weight_total > 0 else 0.0

    metrics = {
        "planning_with_history": planning_with_history,
    }

    if ANALYSIS_MODE:
        planning_no_ctx = weighted_sum_no_ctx / weight_total if weight_total > 0 else 0.0
        metrics["planning_no_ctx"] = planning_no_ctx
        metrics["memory_gain"] = planning_with_history - planning_no_ctx

    print("\n=== FINAL WORKING MEMORY METRICS ===")
    for k, v in metrics.items():
        print(f"{k:24s}: {v:.3f}")

    return metrics, logs


# =========================================================
# TASK
# =========================================================
@kbench.task(name="europa_task_working_memory_history")
def europa_working_memory(llm, steps) -> float:
    try:
        metrics, logs = run_working_memory_episode(llm, steps, task_label="working_memory_history")
        return float(metrics["planning_with_history"])
    except Exception as e:
        print(f"Task-level error: {e}")
        return 0.0


# =========================================================
# RUN
# =========================================================
validate_action_coverage(steps)

df = pd.DataFrame([{"steps": steps}])

print("Loaded total scenarios:", len(TASKS))
print("Testing scenarios:", len(steps))
print("PLAN_HORIZON:", PLAN_HORIZON)
print("HISTORY_WINDOW:", HISTORY_WINDOW)
print("ANALYSIS_MODE:", ANALYSIS_MODE)
print("MEMORY_HIDE_FIELDS:", sorted(MEMORY_HIDE_FIELDS))

results = europa_working_memory.evaluate(
    llm=[kbench.llm],
    evaluation_data=df,
    n_jobs=1,
)

display(results.as_dataframe())